In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

words = open('/Users/raghav/Downloads/names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(stoi)

block_size = 3

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)

random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr   = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte   = build_dataset(words[n2:])

In [4]:
class Linear:
  
  def __init__(self, fan_in, fan_out, bias=True):
    self.weight = torch.randn((fan_in, fan_out), generator=g) / fan_in**0.5
    self.bias = torch.zeros(fan_out) if bias else None
  
  def __call__(self, x):
    self.out = x @ self.weight
    if self.bias is not None:
      self.out += self.bias
    return self.out
  
  def parameters(self):
    return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.momentum = momentum
    self.training = True
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
    # buffers (trained with a running 'momentum update')
    self.running_mean = torch.zeros(dim)
    self.running_var = torch.ones(dim)
  
  def __call__(self, x):
    # calculate the forward pass
    if self.training:
      xmean = x.mean(0, keepdim=True) # batch mean
      xvar = x.var(0, keepdim=True) # batch variance
    else:
      xmean = self.running_mean
      xvar = self.running_var
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    # update the buffers
    if self.training:
      with torch.no_grad():
        self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
        self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []

In [7]:
g = torch.Generator().manual_seed(2147483647)
n_embd = 10
n_hidden = 100
vocab_size = len(stoi)

C = torch.randn(vocab_size,n_embd, generator = g)

layers = [
    Linear((n_embd*block_size),n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,vocab_size,bias=False),BatchNorm1d(vocab_size),
]

parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad = True

47024


In [8]:
max_steps = 200000
batch_size = 32
lossi = []
ud = []

for i in range(max_steps):

    ix = torch.randint(0,Xtr.shape[0],(batch_size,),generator = g)
    Xb , Yb = Xtr[ix], Ytr[ix]

    emb = C[Xb]
    x = emb.view(emb.shape[0],-1)
    for layer in layers:
        x = layer(x)

    loss = F.cross_entropy(x,Yb)

    for p in parameters:
        p.grad = None

    loss.backward()

    lr = 0.1 if i < 150000 else 0.01 # step learning rate decay
    for p in parameters:
        p.data += -lr * p.grad

    if i % 10000 == 0: # print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())
    with torch.no_grad():
        ud.append([((lr*p.grad).std() / p.data.std()).log10().item() for p in parameters])    
    
     

      0/ 200000: 3.6075
  10000/ 200000: 2.3574
  20000/ 200000: 2.0453
  30000/ 200000: 1.8771
  40000/ 200000: 2.1708
  50000/ 200000: 2.2133
  60000/ 200000: 1.8103
  70000/ 200000: 2.0968
  80000/ 200000: 2.2624
  90000/ 200000: 1.8783
 100000/ 200000: 2.2597
 110000/ 200000: 2.1568
 120000/ 200000: 2.1871
 130000/ 200000: 1.9714
 140000/ 200000: 1.7175
 150000/ 200000: 1.8564
 160000/ 200000: 1.9988
 170000/ 200000: 1.9490
 180000/ 200000: 2.2312
 190000/ 200000: 1.9662


In [10]:
for layer in layers:
    layer.training = False

with torch.no_grad():
    emb = C[Xtr]
    x = emb.view(Xtr.shape[0],-1)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x,Ytr)


print(loss)

tensor(2.0003)


In [13]:
for layer in layers:
    layer.training = False

with torch.no_grad():
    emb = C[Xdev]
    x = emb.view(Xdev.shape[0],-1)
    for layer in layers:
        x = layer(x)
    loss = F.cross_entropy(x,Ydev)


print(loss)

tensor(2.0792)


In [16]:
for _ in range(20):

    out = []
    context = [0] * block_size
    while True:

        emb = C[torch.tensor([context])]
        x = emb.view(emb.shape[0],-1)
        for layer in layers:
            x = layer(x)

        logits = x
        probs = F.softmax(logits, dim = 1)
        ix = torch.multinomial(probs,num_samples = 1,generator = g).item()

        context = context[1:] + [ix]
        out.append(ix)

        if ix == 0:
            break
            
    print(''.join(itos[i] for i in out))

caron.
indysmaeli.
solly.
ain.
kaylah.
saradoraige.
ars.
cadessia.
salian.
mahelen.
lay.
camfee.
quet.
ercere.
loreyan.
alce.
jaida.
lamia.
wolly.
saheddori.
